# Notebook 08b: Coherent Execution Validation (Family C)

Family C instances (C1, C2, C3) are small enough for coherent DQI statevector execution. This notebook validates encoding and execution mode choices on this subset.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    from notebooks._helpers import (
        repo_root,
        ensure_numeric_columns,
        load_matrix_report,
        load_matrix_summary,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        parse_metric_availability,
        ensure_optional_columns,
        placeholder_plot,
        unavailable_or_numeric,
        unavailable_panel_table,
    )
except ModuleNotFoundError:
    from _helpers import (
        repo_root,
        ensure_numeric_columns,
        load_matrix_report,
        load_matrix_summary,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        parse_metric_availability,
        ensure_optional_columns,
        placeholder_plot,
        unavailable_or_numeric,
        unavailable_panel_table,
    )

ROOT = repo_root()

pd.set_option("display.max_columns", 120)
from src.notebook_utils import FilterPipeline


In [2]:
run_manifest = load_run_manifest(ROOT / "results", required=False)
metrics_master = load_metrics_master(ROOT / "results", required=False)

results_root = ROOT / "results"
master_parquet_path = results_root / "metrics_master.parquet"
master_json_path = results_root / "metrics_master.json"
master_artifacts_available = master_parquet_path.exists() or master_json_path.exists()

source_details = {
    "preferred": "saved benchmark_*feat.json paired-comparison payloads",
    "fallback": "matrix_a_summary.csv + matrix_a_report.json",
    "active": "none",
    "stage_filter": "not_evaluated",
    "family_filter": "not_evaluated",
    "master_artifacts_available": bool(master_artifacts_available),
}

benchmark_paths = sorted(results_root.glob("benchmark_*feat.json"))
benchmark_records = []
for benchmark_path in benchmark_paths:
    try:
        payload = json.loads(benchmark_path.read_text())
    except Exception:
        continue
    if isinstance(payload, dict):
        benchmark_records.append(payload)

paired_outputs = []
for record in benchmark_records:
    paired = record.get("stage1_paired_encoding")
    recommendation = record.get("encoding_backend_recommendation", {})
    if not isinstance(paired, dict):
        continue
    paired_outputs.append({
        "instance_id": record.get("problem", {}).get("n_features"),
        "instance_label": f"P{record.get('problem', {}).get('n_features')}" if record.get("problem", {}).get("n_features") is not None else "unknown",
        "paired": paired,
        "recommendation": recommendation,
    })

focus = pd.DataFrame()
selection_mode = "legacy_fixed_instance_fallback"
if not metrics_master.empty:
    focus = metrics_master.copy()
    focus = ensure_run_status_norm(focus)
    if "stage" in focus.columns:
        focus = focus[focus["stage"].fillna("benchmark_matrix").astype(str).isin(["benchmark_matrix", "matrix_a"])].copy()
    if "family" in focus.columns:
        focus = focus[focus["family"].fillna("P").astype(str).eq("P")].copy()
    if "execution_mode" in focus.columns and "execution_mode_norm" not in focus.columns:
        focus["execution_mode_norm"] = focus["execution_mode"].astype(str).str.strip().str.lower()
    if "run_status_norm" in focus.columns:
        focus = focus[focus["run_status_norm"] == "completed"].copy()
    selection_mode = "stage_family_manifest_aligned"
else:
    legacy_summary = load_matrix_summary("matrix_a", ROOT / "results")
    if not legacy_summary.empty:
        focus = legacy_summary.copy()
        focus = ensure_run_status_norm(focus)
        if "execution_mode" in focus.columns and "execution_mode_norm" not in focus.columns:
            focus["execution_mode_norm"] = focus["execution_mode"].astype(str).str.strip().str.lower()
        if "run_status_norm" in focus.columns:
            focus = focus[focus["run_status_norm"] == "completed"].copy()
        selection_mode = "legacy_fixed_instance_fallback"

source_details["active"] = "saved_benchmark_json" if paired_outputs else ("metrics_master" if not metrics_master.empty else "matrix_a_summary")
display(pd.DataFrame([source_details]))



# --- Family C specific loading (from original Cell 16) ---
# Load Family C alpha validation data
family_c_df = pd.DataFrame()
family_c_reason = None

if metrics_master.empty:
    family_c_reason = "metrics_master is unavailable"
else:
    family_c_df = metrics_master.copy()
    family_c_df = ensure_run_status_norm(family_c_df)
    
    # Primary filter: look for stage=alpha_validation OR fallback to Family C matrix_b
    if "stage" in family_c_df.columns:
        alpha_validation_rows = family_c_df[family_c_df["stage"].astype(str).str.lower() == "alpha_validation"].copy()
        if not alpha_validation_rows.empty:
            family_c_df = alpha_validation_rows
        else:
            # Fallback: Use Family C rows from matrix_b (coherent validation data)
            if "family" in family_c_df.columns and "matrix" in family_c_df.columns:
                family_c_rows = family_c_df[
                    (family_c_df["family"].astype(str).eq("C")) &
                    (family_c_df["matrix"].astype(str).eq("matrix_b"))
                ].copy()
                if not family_c_rows.empty:
                    family_c_df = family_c_rows
                else:
                    family_c_df = pd.DataFrame()
            else:
                family_c_df = pd.DataFrame()
    
    if "family" in family_c_df.columns:
        family_c_df = family_c_df[family_c_df["family"].astype(str).eq("C")].copy()
    if "run_status_norm" in family_c_df.columns:
        family_c_df = family_c_df[family_c_df["run_status_norm"] == "completed"].copy()
    
    # Normalize execution_mode
    if "execution_mode" in family_c_df.columns and "execution_mode_norm" not in family_c_df.columns:
        family_c_df["execution_mode_norm"] = family_c_df["execution_mode"].astype(str).str.strip().str.lower()
    
    if family_c_df.empty:
        family_c_reason = "No completed Family C rows found in alpha_validation stage or matrix_b"

if family_c_reason:
    display(unavailable_panel_table(
        panel="Family C coherent validation",
        reason=family_c_reason,
    ))
else:
    # Summary of available data
    summary_data = []
    for enc in family_c_df["encoding"].unique() if "encoding" in family_c_df.columns else []:
        for mode in ["coherent", "mixture"]:
            subset = family_c_df[
                (family_c_df["encoding"] == enc) & 
                (family_c_df["execution_mode_norm"] == mode)
            ] if "execution_mode_norm" in family_c_df.columns else pd.DataFrame()
            summary_data.append({
                "encoding": enc,
                "execution_mode": mode,
                "completed_rows": len(subset),
                "instances": ", ".join(sorted(subset["instance_id"].unique())) if "instance_id" in subset.columns else "unknown",
            })
    display(Markdown("### Family C data availability"))
    display(pd.DataFrame(summary_data))

,preferred,fallback,active,stage_filter,family_filter,master_artifacts_available
0,saved benchmark_*feat.json paired-comparison p...,matrix_a_summary.csv + matrix_a_report.json,saved_benchmark_json,not_evaluated,not_evaluated,True


### Family C data availability

,encoding,execution_mode,completed_rows,instances
0,wht_exact,coherent,12,"C1, C2"
1,wht_exact,mixture,18,"C1, C2, C3"
2,ilp_derived_exact,coherent,6,C2
3,ilp_derived_exact,mixture,6,C2


## Coherent Execution Validation (Family C)

Family C instances (C1, C2, C3) are small enough for coherent DQI statevector simulation. This section validates whether coherent mode preserves the quality advantages observed in mixture mode, and compares ILP-derived vs WHT-exact encodings in coherent execution.

In [3]:
# Family C data already loaded and displayed above — skip redundant reload
# (coherent validation uses family_c_df from the initial load cell)

### ILP-derived coherent vs mixture on Family C

Compare coherent and mixture execution modes for ILP-derived encoding (C2 instance, which is derived from a tiny pricing problem).

In [4]:
# ILP-derived coherent vs mixture comparison
ilp_c_df = pd.DataFrame()
ilp_comparison_reason = None

if family_c_df.empty:
    ilp_comparison_reason = "Family C data unavailable"
elif "encoding" not in family_c_df.columns:
    ilp_comparison_reason = "encoding column missing"
else:
    ilp_c_df = family_c_df[family_c_df["encoding"].astype(str).str.contains("ilp", case=False)].copy()
    if ilp_c_df.empty:
        ilp_comparison_reason = "No ILP-derived encoding rows found in Family C"

if ilp_comparison_reason:
    display(unavailable_panel_table(
        panel="ILP coherent vs mixture",
        reason=ilp_comparison_reason,
    ))
else:
    # Display table with key metrics
    display_cols = ["instance_id", "decoder", "alpha_mode", "execution_mode", 
                    "best_sampled_F", "top1_regret", "postselection_success"]
    available_cols = [c for c in display_cols if c in ilp_c_df.columns]
    
    ilp_display = ilp_c_df[available_cols].copy()
    for col in ["best_sampled_F", "top1_regret", "postselection_success"]:
        if col in ilp_display.columns:
            ilp_display[col] = pd.to_numeric(ilp_display[col], errors="coerce")
    
    ilp_display = ilp_display.sort_values(
        [c for c in ["instance_id", "decoder", "alpha_mode", "execution_mode"] if c in ilp_display.columns]
    ).reset_index(drop=True)
    
    display(Markdown("**ILP-derived encoding results (Family C):**"))
    display(ilp_display)
    
    # Compute paired delta (coherent - mixture)
    if "execution_mode_norm" in ilp_c_df.columns:
        coherent_ilp = ilp_c_df[ilp_c_df["execution_mode_norm"] == "coherent"].copy()
        mixture_ilp = ilp_c_df[ilp_c_df["execution_mode_norm"] == "mixture"].copy()
        
        pair_cols = [c for c in ["instance_id", "decoder", "alpha_mode"] if c in coherent_ilp.columns]
        
        if not coherent_ilp.empty and not mixture_ilp.empty and pair_cols:
            merged = coherent_ilp.merge(mixture_ilp, on=pair_cols, suffixes=("_coh", "_mix"), how="inner")
            
            if not merged.empty:
                delta_rows = []
                for _, row in merged.iterrows():
                    delta_row = {c: row[c] for c in pair_cols}
                    for metric in ["best_sampled_F", "top1_regret", "postselection_success"]:
                        coh_val = pd.to_numeric(row.get(f"{metric}_coh"), errors="coerce")
                        mix_val = pd.to_numeric(row.get(f"{metric}_mix"), errors="coerce")
                        if pd.notna(coh_val) and pd.notna(mix_val):
                            delta_row[f"delta_{metric}"] = float(coh_val - mix_val)
                        else:
                            delta_row[f"delta_{metric}"] = np.nan
                    delta_rows.append(delta_row)
                
                delta_ilp_df = pd.DataFrame(delta_rows)
                display(Markdown("\n**Paired delta (coherent - mixture) for ILP-derived:**"))
                display(delta_ilp_df)

**ILP-derived encoding results (Family C):**

,instance_id,decoder,alpha_mode,execution_mode,best_sampled_F,top1_regret,postselection_success
0,C2,bp1,heuristic,coherent,600.0,0.0,1.0
1,C2,bp1,heuristic,mixture,600.0,0.0,1.0
2,C2,bp1,paper,coherent,600.0,0.0,1.0
3,C2,bp1,paper,mixture,600.0,0.0,1.0
4,C2,bp1,uniform,coherent,600.0,0.0,1.0
5,C2,bp1,uniform,mixture,600.0,0.0,1.0
6,C2,oracle,heuristic,coherent,600.0,0.0,1.0
7,C2,oracle,heuristic,mixture,600.0,0.0,1.0
8,C2,oracle,paper,coherent,600.0,0.0,1.0
9,C2,oracle,paper,mixture,600.0,0.0,1.0



**Paired delta (coherent - mixture) for ILP-derived:**

,instance_id,decoder,alpha_mode,delta_best_sampled_F,delta_top1_regret,delta_postselection_success
0,C2,oracle,uniform,0.0,0.0,0.000000e+00
1,C2,bp1,uniform,0.0,0.0,0.000000e+00
2,C2,oracle,paper,0.0,0.0,2.220446e-16
3,C2,bp1,paper,0.0,0.0,2.220446e-16
4,C2,oracle,heuristic,0.0,0.0,4.440892e-16
5,C2,bp1,heuristic,0.0,0.0,4.440892e-16


### Encoding comparison in coherent mode (Family C)

Compare ILP-derived (C2) vs WHT-exact (C1, C3) encodings in coherent execution mode. This addresses: *Does the ILP advantage hold in coherent execution?*

In [5]:
# Encoding comparison in coherent mode
coherent_c_df = pd.DataFrame()
encoding_comparison_reason = None

if family_c_df.empty:
    encoding_comparison_reason = "Family C data unavailable"
elif "execution_mode_norm" not in family_c_df.columns:
    encoding_comparison_reason = "execution_mode column missing"
else:
    coherent_c_df = family_c_df[family_c_df["execution_mode_norm"] == "coherent"].copy()
    if coherent_c_df.empty:
        encoding_comparison_reason = "No coherent execution rows found in Family C"

if encoding_comparison_reason:
    display(unavailable_panel_table(
        panel="Encoding comparison (coherent mode)",
        reason=encoding_comparison_reason,
    ))
else:
    # Aggregate by encoding
    agg_metrics = ["best_sampled_F", "top1_regret", "postselection_success"]
    for col in agg_metrics:
        if col in coherent_c_df.columns:
            coherent_c_df[col] = pd.to_numeric(coherent_c_df[col], errors="coerce")
    
    # Summary by encoding
    if "encoding" in coherent_c_df.columns:
        enc_summary = []
        for enc in coherent_c_df["encoding"].unique():
            enc_data = coherent_c_df[coherent_c_df["encoding"] == enc]
            row = {
                "encoding": enc,
                "instances": ", ".join(sorted(enc_data["instance_id"].unique())) if "instance_id" in enc_data.columns else "unknown",
                "n_rows": len(enc_data),
            }
            for metric in agg_metrics:
                if metric in enc_data.columns:
                    row[f"mean_{metric}"] = enc_data[metric].mean()
                    row[f"zero_regret_count"] = (enc_data["top1_regret"].abs() < 1e-9).sum() if metric == "top1_regret" else None
            enc_summary.append(row)
        
        enc_summary_df = pd.DataFrame(enc_summary)
        display(Markdown("**Coherent mode: encoding comparison summary:**"))
        display(enc_summary_df)
        
        # Detailed view by instance
        display_cols = ["instance_id", "encoding", "alpha_mode", "best_sampled_F", "top1_regret", "postselection_success"]
        available_cols = [c for c in display_cols if c in coherent_c_df.columns]
        
        coherent_detail = coherent_c_df[available_cols].copy()
        coherent_detail = coherent_detail.sort_values(
            [c for c in ["encoding", "instance_id", "alpha_mode"] if c in coherent_detail.columns]
        ).reset_index(drop=True)
        
        display(Markdown("\n**Detailed coherent results by instance and encoding:**"))
        display(coherent_detail)

**Coherent mode: encoding comparison summary:**

,encoding,instances,n_rows,mean_best_sampled_F,zero_regret_count,mean_top1_regret,mean_postselection_success
0,wht_exact,"C1, C2",12,302.0,None,0.0,1.0
1,ilp_derived_exact,C2,6,600.0,None,0.0,1.0



**Detailed coherent results by instance and encoding:**

,instance_id,encoding,alpha_mode,best_sampled_F,top1_regret,postselection_success
0,C2,ilp_derived_exact,heuristic,600.0,0.0,1.0
1,C2,ilp_derived_exact,heuristic,600.0,0.0,1.0
2,C2,ilp_derived_exact,paper,600.0,0.0,1.0
3,C2,ilp_derived_exact,paper,600.0,0.0,1.0
4,C2,ilp_derived_exact,uniform,600.0,0.0,1.0
5,C2,ilp_derived_exact,uniform,600.0,0.0,1.0
6,C1,wht_exact,heuristic,4.0,0.0,1.0
7,C1,wht_exact,heuristic,4.0,0.0,1.0
8,C1,wht_exact,paper,4.0,0.0,1.0
9,C1,wht_exact,paper,4.0,0.0,1.0


In [6]:
# Coherent execution verdict
display(Markdown("### Coherent execution verdict (Family C)"))

if family_c_df.empty:
    display(Markdown("- Coherent validation data unavailable."))
else:
    # Check if any coherent runs achieve zero regret
    coherent_df = family_c_df[family_c_df["execution_mode_norm"] == "coherent"].copy() if "execution_mode_norm" in family_c_df.columns else pd.DataFrame()
    
    if coherent_df.empty:
        display(Markdown("- No coherent execution results available."))
    else:
        coherent_df["top1_regret"] = pd.to_numeric(coherent_df["top1_regret"], errors="coerce")
        
        # Per-instance verdict
        verdict_rows = []
        for instance_id in coherent_df["instance_id"].unique() if "instance_id" in coherent_df.columns else []:
            inst_data = coherent_df[coherent_df["instance_id"] == instance_id]
            has_zero_regret = (inst_data["top1_regret"].abs() < 1e-9).any()
            min_regret = inst_data["top1_regret"].min()
            encoding = inst_data["encoding"].iloc[0] if "encoding" in inst_data.columns else "unknown"
            verdict_rows.append({
                "instance_id": instance_id,
                "encoding": encoding,
                "verdict": "\u2713" if has_zero_regret else "\u2717",
                "min_top1_regret": min_regret,
            })
        
        verdict_df = pd.DataFrame(verdict_rows)
        display(Markdown("**Per-instance verdict: \u2713 if top1_regret = 0 for at least one (decoder, alpha) config in coherent mode**"))
        display(verdict_df)
        
        # Summary
        passing_instances = verdict_df[verdict_df["verdict"] == "\u2713"]["instance_id"].tolist()
        ilp_coherent = coherent_df[coherent_df["encoding"].astype(str).str.contains("ilp", case=False)]
        ilp_zero_regret = (ilp_coherent["top1_regret"].abs() < 1e-9).any() if not ilp_coherent.empty else False
        
        if ilp_zero_regret:
            display(Markdown(
                f"- **ILP-derived encoding achieves zero regret in coherent mode** on Family C, "
                f"matching the zero-regret result observed in mixture mode on Family P."
            ))
        else:
            ilp_min = ilp_coherent["top1_regret"].min() if not ilp_coherent.empty else np.nan
            display(Markdown(
                f"- ILP-derived encoding shows min top1_regret = {ilp_min:.4f} in coherent mode. "
                f"This may reflect finite-size effects in the smaller Family C instances."
            ))

### Coherent execution verdict (Family C)

**Per-instance verdict: ✓ if top1_regret = 0 for at least one (decoder, alpha) config in coherent mode**

,instance_id,encoding,verdict,min_top1_regret
0,C1,wht_exact,✓,0.0
1,C2,wht_exact,✓,0.0


- **ILP-derived encoding achieves zero regret in coherent mode** on Family C, matching the zero-regret result observed in mixture mode on Family P.

### Scope note: Family P coherent execution

Family P coherent DQI execution requires statevector simulation of the joint |t>|e>|s> space, where:
- |t> = weight register (log2(ell+1) qubits)
- |e> = error register (m qubits for m parity terms)
- |s> = solution register (n qubits)

For Family P instances:
- P3: n=6, m=12 (k=12 Fourier terms), yielding 2^20 = 1M+ states at ell=3
- P4: n=8, m=12, yielding 2^22 = 4M+ states
- P5: n=10, m=15, yielding 2^27 = 134M+ states

These exceed the repository's statevector simulation budget (4096 states for exact coherent simulation).

**Coherent validation is therefore performed on the smaller Family C instances** where statevector simulation is tractable (C1: 1024 states, C2: 4096 states at the cap).

Extending coherent DQI execution to Family P scale is a natural target for:
- Tensor-network simulation methods
- Near-term quantum hardware with 30+ qubits
- Approximate statevector methods with controlled error bounds

This represents a concrete direction for scaling validation beyond the current benchmark scope.

## See also

Paired encoding comparison for Family P is in [Notebook 08a](08a_paired_encoding_comparison.ipynb).